In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cmocean as cmo
import cartopy.crs as ccrs

from utils import get_shelf_edge, subplots_structure, nearest, prepare_eddies2plot, set_figure, make_map
from load_profiles import load_vertical_profiles

In [2]:
proj=ccrs.PlateCarree()
extent=[-47.5, -37.5, -28, -20.62] # max lat fits the gebco data I have for sheld egde contour
secGloc = (-43.072, -23.033)
secGloc = (-43.072, -23.033)
idate = '2022-03-01'
edate = '2022-04-01'
lon_range = extent[:2]
lat_range = extent[2:]
time_range = pd.date_range(idate, edate, freq='1D')

depfile = '../data/gebco_2019.nc'
# before plot the map, get the sheld edge contour
shelf_edge = get_shelf_edge(depfile=depfile, extent=extent)

datadir = '../data/sanagu2022/AVHRR/'
ds = xr.open_mfdataset(f"{datadir}/*.nc")


In [3]:

# selecting only valid flags data
ds = ds.where(ds.quality_level > 3, drop=True)

# loading eddies
ds_locs = xr.open_dataset(f"{datadir}../eddies_sanagu2022_daily.nc")

# weekly avg maps
ds_weekly = ds.resample(time='7D').mean()

KeyError: 'Indexing with a boolean dask array is not allowed. This will result in a dask array of unknown shape. Such arrays are unsupported by Xarray.Please compute the indexer first using .compute()'

In [ ]:
dict_casts = load_vertical_profiles()

In [ ]:
print(f"Plotting 3-days composites for SANAGU14 and SANAGU15")
set_figure(analysis=None) # applying rcParams for analysis

# loading AVISO ssh for geostrophic currents
ds_aviso = xr.open_dataset(f"{datadir}/../dataset-duacs-nrt-global-merged-allsat-phy-l4_20220101_20220401.nc")

# selecting days for eddies
ds_eddies = []
for date in time_range[[9, 10, 11]]:
    _ = ds_locs.where(ds_locs['time'] == date, drop=True)
    ds_eddies.append(_)

# get 3-days average of geostrophic currents
ds_geostrophic_before = ds_aviso.sel(time=time_range[[9,10,11]]).mean(dim='time')

# preparing longitude from eddies to plot
ds_eddies_before = prepare_eddies2plot(xr.concat(ds_eddies, dim='obs'))

ds_eddies = []
for date in time_range[[27, 28, 29]]:
    _ = ds_locs.where(ds_locs['time'] == date, drop=True)
    ds_eddies.append(_)

# get 3-days average of geostrophic currents
ds_geostrophic_after = ds_aviso.sel(time=time_range[[27, 28, 29]]).mean(dim='time')

# preparing longitude from eddies to plot
ds_eddies_after = prepare_eddies2plot(xr.concat(ds_eddies, dim='obs'))

ds_daily = ds.resample(time='1D').mean()
# -- sea surface temperature in Celsius
ds_plot = (ds_daily.sea_surface_temperature - 273.15).sel(lon=slice(-47.5, -37.5), lat=slice(-20, -28), )

# selecting days to take a composite view after and before cold front
ds_composite_before = ds_daily.sel(time=['2022-03-10', 
                                        '2022-03-11', 
                                        '2022-03-12']).mean(dim='time')
ds_composite_before = (ds_composite_before.sea_surface_temperature - 273.15).sel(lon=slice(-47.5, -37.5), lat=slice(-20, -28), )

ds_composite_after = ds_daily.sel(time=['2022-03-28', 
                                        '2022-03-29', 
                                        '2022-03-30']).mean(dim='time')
ds_composite_after = (ds_composite_after.sea_surface_temperature - 273.15).sel(lon=slice(-47.5, -37.5), lat=slice(-20, -28), )

# plotting composites
fig = plt.figure(figsize=(20/2.54, 16/2.54)) # proportion for paper (15/2.54, 8/2.54)

ax_before = fig.add_subplot(1, 2, 1, projection=ccrs.PlateCarree())
ax_before = make_map(ax=ax_before, extent=extent)
ax_before.set_extent(extent)

ax_after = fig.add_subplot(1, 2, 2, projection=ccrs.PlateCarree())
ax_after = make_map(ax=ax_after, extent=extent)
ax_after.set_extent(extent)

ax_before.set_title('[a] 10-12, March [AVHRR]', loc='left', fontsize=9, y=.89)
ax_after.set_title('[b] 28-30, March [AVHRR]', loc='left', fontsize=9, y=.89)

pc1 = ds_composite_before.plot.pcolormesh(ax=ax_before,
                                        vmin=22,
                                        vmax=30,
                                        cmap='coolwarm',
                                        add_colorbar=False)

pc2 = ds_composite_after.plot.pcolormesh(ax=ax_after,
                                        vmin=22,
                                        vmax=30,
                                        cmap='coolwarm',
                                        add_colorbar=False)

# adding geostrophic vectors
ax_before.quiver(ds_geostrophic_before['longitude'], ds_geostrophic_before['latitude'], ds_geostrophic_before['ugos'], ds_geostrophic_before['vgos'], scale=25, alpha=.5)
ax_after.quiver(ds_geostrophic_after['longitude'], ds_geostrophic_after['latitude'], ds_geostrophic_after['ugos'], ds_geostrophic_after['vgos'], scale=25, alpha=.5)

# adding eddies contour

# inserting anticyclonic eddies contour
for ax,eddy in zip([ax_before, ax_after], [ds_eddies_before, ds_eddies_after]):
    for i in range(eddy.dims['obs']):        
        _ = eddy.isel(obs=i)
        _anti = _.where(_['eddy'] == 'anticyclonic', drop=True)
        _cycl = _.where(_['eddy'] == 'cyclonic', drop=True)

        ax.plot(_anti['effective_contour_longitude'], _anti['effective_contour_latitude'], 'r', lw=1)    
        ax.plot(_cycl['effective_contour_longitude'], _cycl['effective_contour_latitude'], 'b', lw=1)

# adding marker and shelf edge contour
for ax in [ax_before, ax_after]:
    ax.scatter(secGloc[0], secGloc[1], marker='*', s=30, c='k', zorder=5)
    ax.plot(shelf_edge['lon'], shelf_edge['lat'], ls='--', c='k', lw='0.5')

    # plotting
    for cast,values in dict_casts.items():
        prof = values['prof']
        str_time = values['time'].strftime("%m/%d")
        xlon = values['xlon']
        ylat = values['ylat']
        c = 'k'
        ax.scatter(xlon, ylat, c=c, s=50)

cb = plt.colorbar(pc1, orientation="horizontal", pad=0.05)
cb.set_label(label='sea surface temperature [$^{\circ}$C]', size=10)
cb.ax.tick_params(labelsize='small')

cb = plt.colorbar(pc2, orientation="horizontal", pad=0.05)
cb.set_label(label='sea surface temperature [$^{\circ}$C]', size=10)
cb.ax.tick_params(labelsize='small')

plt.tight_layout()
plt.subplots_adjust(top=0.952,
                    bottom=0.094,
                    left=0.083,
                    right=0.96,
                    hspace=0.2,
                    wspace=0.251)

fig.patch.set_alpha(1.0)
plt.savefig('../figures/manuscript/00_manuscript_supp_sst.png', bbox_inches="tight", dpi=300)